# AI온캠 팀 데이터셋 전체 학습

Colab에서 T4 GPU를 선택하고 위에서부터 실행합니다. 팀 Drive는 읽기 전용으로 연결합니다. 파일 선택 단계에서 AI온캠_전체학습_입력.zip을 올립니다. 파일 목록은 지금 감사한 데이터 배치 기준입니다.

In [ ]:
from pathlib import Path
from google.colab import drive, files
import json

drive.mount('/content/drive', readonly=True)
mount = Path('/content/drive')
roots = [mount / 'MyDrive']
if (mount / 'Shareddrives').is_dir():
    roots.extend(p for p in (mount / 'Shareddrives').iterdir() if p.is_dir())
team = next((p for r in roots for p in (r, r / '온캠퍼스_6조', *r.glob('*/온캠퍼스_6조'))
             if p.name == '온캠퍼스_6조' and p.is_dir()), None)
if team is None:
    raise FileNotFoundError('팀 Drive 계정으로 연결한 뒤 이 셀을 다시 실행하세요.')
print('팀 폴더:', team)


In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.48.0'])


In [ ]:
from google.colab import files
from zipfile import ZipFile
import sys
inputs = Path('/content/aioncamp_full_training/inputs')
inputs.mkdir(parents=True,exist_ok=True)
if not (inputs/'manifest.csv').is_file():
    uploaded = files.upload()
    with ZipFile('AI온캠_전체학습_입력.zip') as z:
        for name in ('fit_team_dataset.py','manifest.csv'):
            (inputs / name).write_bytes(z.read(name))
    del uploaded
sys.path.insert(0, str(inputs))


In [ ]:
import torch
from fit_team_dataset import train
from zipfile import ZIP_DEFLATED
assert torch.cuda.is_available(), 'Colab에서 GPU 런타임을 선택하세요.'
output = inputs.parent
train(team, inputs/'manifest.csv', output, batch_size=32)
export = output/'ai_oncamp_full_training.zip'
with ZipFile(export,'w',ZIP_DEFLATED) as z:
    for name in ('character_head.npz','full_training_report.json'):
        z.write(output/name,name)
files.download(str(export))
